In [1]:
import pdal
import geopandas as gpd
import json
import pandas as pd
import numpy as np
from shapely import MultiPoint
import shapely
from buildingregulariser import regularize_geodataframe
import warnings
import math
import os

(PDAL Error) Can't load library /Users/fernandogomes/miniconda3/envs/pdal/lib/libpdal_plugin_reader_arrow.dylib: Failed to load "/Users/fernandogomes/miniconda3/envs/pdal/lib/libpdal_plugin_reader_arrow.dylib": dlopen(/Users/fernandogomes/miniconda3/envs/pdal/lib/libpdal_plugin_reader_arrow.dylib, 0x0002): Symbol not found: _BrotliDefaultAllocFunc
  Referenced from: <AE78991A-FDBC-355F-AE1F-B456D3A59B38> /Users/fernandogomes/miniconda3/envs/pdal/lib/libbrotlienc.1.1.0.dylib
  Expected in:     <751D82F8-ADA1-3A74-8E9B-C1B682F93468> /Users/fernandogomes/miniconda3/envs/pdal/lib/libbrotlicommon.1.0.7.dylib(PDAL Error) Can't load library /Users/fernandogomes/miniconda3/envs/pdal/lib/libpdal_plugin_writer_arrow.dylib: Failed to load "/Users/fernandogomes/miniconda3/envs/pdal/lib/libpdal_plugin_writer_arrow.dylib": dlopen(/Users/fernandogomes/miniconda3/envs/pdal/lib/libpdal_plugin_writer_arrow.dylib, 0x0002): Symbol not found: _BrotliDefaultAllocFunc
  Referenced from: <AE78991A-FDBC-355F-A

In [2]:
RESULT_FOLDER = '/Users/fernandogomes/dev/LiDAR_produtos'

In [3]:
resolution = 0.5
gdf_distritos = gpd.read_file('data/SIRGAS_GPKG_distrito.gpkg')
gdf_articulacao_17_20 = gpd.read_file("zip://data/SIRGAS_SHP_quadriculamdt.zip!/SIRGAS_SHP_quadriculamdt/")
gdf_articulacao_24 = gpd.read_file('results/folhas_sp_cortada.gpkg')
gdf_articulacao_24.to_crs(epsg=31983, inplace=True)

In [4]:
gdf_folhas = gdf_articulacao_24.overlay(gdf_distritos[gdf_distritos.ds_nome == 'BRAS'])

In [5]:
folhas = [f'{RESULT_FOLDER}/2024/BHM/BHM-{nome}-2024-50cm.tiff' for nome in gdf_folhas.nome.to_list()]

In [6]:
' '.join(folhas)

'/Users/fernandogomes/dev/LiDAR_produtos/2024/BHM/BHM-Y-C-VI-2-NO-D-II-6-2024-50cm.tiff /Users/fernandogomes/dev/LiDAR_produtos/2024/BHM/BHM-Y-C-VI-2-NO-D-II-3-2024-50cm.tiff /Users/fernandogomes/dev/LiDAR_produtos/2024/BHM/BHM-Y-C-VI-2-NO-B-IV-6-2024-50cm.tiff /Users/fernandogomes/dev/LiDAR_produtos/2024/BHM/BHM-Y-C-VI-2-NE-C-I-4-2024-50cm.tiff /Users/fernandogomes/dev/LiDAR_produtos/2024/BHM/BHM-Y-C-VI-2-NE-C-I-1-2024-50cm.tiff /Users/fernandogomes/dev/LiDAR_produtos/2024/BHM/BHM-Y-C-VI-2-NE-A-III-4-2024-50cm.tiff /Users/fernandogomes/dev/LiDAR_produtos/2024/BHM/BHM-Y-C-VI-2-NE-C-I-5-2024-50cm.tiff /Users/fernandogomes/dev/LiDAR_produtos/2024/BHM/BHM-Y-C-VI-2-NE-C-I-2-2024-50cm.tiff /Users/fernandogomes/dev/LiDAR_produtos/2024/BHM/BHM-Y-C-VI-2-NE-A-III-5-2024-50cm.tiff'

In [7]:
!gdalbuildvrt temp/BHM-bras.vrt {' '.join(folhas)}

0...10...20...30...40...50...60...70...80...90...100 - done.


In [8]:
!gdal_translate temp/BHM-bras.vrt temp/BHM-bras.tiff

Input file size is 6458, 6994
0...10...20...30...40...50...60...70...80...90...100 - done.


In [9]:
# def laz_pipeline(resolution):
#     return [
#         {
#             "type":"readers.las",
#             "filename":f"/Users/fernandogomes/dev/LiDAR_produtos/2024/LiDAR-subs/10-BRAS-buildings-50cm.laz"
#         },
#         {
#             "type":"filters.dbscan",
#             "min_points":5,
#             "eps": 0.5 * math.sqrt(2),
#             "dimensions":"X,Y,Z"
#         },
#     ]

In [10]:
def laz_pipeline(resolution):
    return [
        {
            "type":"readers.gdal",
            "filename":f"temp/BHM-bras.tiff"
        },
        {
            "type":"filters.ferry",
            "dimensions":"band_1 => Z"
        },
        {
            "type":"filters.dbscan",
            "min_points":5,
            "eps": 0.5 * math.sqrt(2),
            "dimensions":"X,Y,Z"
        },
    ]

In [11]:
agg = {
    'coords':list,  
    'Z':['count', 'median', 'std', 'max'], 
    # 'Intensity':'median', 
    # 'Red':'median',
    # 'Green':'median',
    # 'Blue':'median'  
}

columns = {
    ('coords', 'list'):'coords',
    ('Z', 'count'):'z_count',
    ('Z', 'median'):'z_median',
    ('Z', 'std'):'z_std',
    ('Z', 'max'):'z_max',
    # ('Intensity', 'median'):'intensity_median',
    # ('Red', 'median'):'red_median',
    # ('Green', 'median'):'green_median',
    # ('Blue', 'median'):'blue_median',
}

In [12]:
laz = laz_pipeline(resolution)
pipeline = pdal.Pipeline(json.dumps(laz))
n_points = pipeline.execute()
print(f'Pipeline selected {n_points} points')

Pipeline selected 45167252 points


In [13]:
arr = pipeline.arrays[0]
df = pd.DataFrame(arr)
df = df[df.ClusterID > 0].reset_index()
df.loc[:, 'coords'] = list(np.dstack([df.X, df.Y])[0])
df['Z'] = df.groupby(['X', 'Y'])['Z'].transform('max')
df.drop_duplicates(subset=['X', 'Y'], keep='last', inplace=True)
df = df[(df.Z > 2.0) & (df.Z < 200.0)].reset_index()

In [14]:
df = df[df.groupby("ClusterID")["ClusterID"].transform("count") > 16]

In [15]:
df[df.ClusterID > 0].groupby('ClusterID').agg(agg)

coords     Z             \
                                                        list count     median   
ClusterID                                                                       
1          [[335182.75, 7396753.25], [335183.25, 7396753....  3336   9.227093   
2          [[335185.75, 7396753.25], [335186.25, 7396753....  1535  12.406150   
3          [[335193.75, 7396753.25], [335194.25, 7396753....   252  20.799744   
4          [[335203.25, 7396753.25], [335203.75, 7396753....   111  24.681206   
5          [[335209.75, 7396753.25], [335210.25, 7396753....  1121  19.095282   
...                                                      ...   ...        ...   
282745     [[333863.25, 7393259.25], [333863.75, 7393259....    64   2.494175   
282746     [[333966.25, 7393258.75], [333966.75, 7393258....    32   7.285889   
282750     [[334098.75, 7393259.25], [334098.25, 7393258....    22  18.335293   
282760     [[333352.25, 7393258.75], [333352.75, 7393258....    30   7.112373   
282769     [[333558.75, 7393258.25], [333559.25, 7393258....    18  51.072018   

                                
                std        max  
ClusterID                       
1          0.460550  10.333432  
2          0.669694  13.726165  
3          0.201013  21.241028  
4          0.133715  25.016811  
5          0.267946  20.301847  
...             ...        ...  
282745     0.302380   3.122104  
282746     0.211199   7.690427  
282750     0.445398  18.757166  
282760     0.323198   7.506892  
282769     0.072550  51.298168  

[45704 rows x 5 columns]

In [16]:
df_agg = df.groupby('ClusterID').agg(agg)
df_agg.columns = df_agg.columns.to_flat_index()
df_agg.rename(columns=columns, inplace=True)
df_agg.loc[:, 'geometry'] = df_agg.coords.apply(MultiPoint)
gdf_agg = gpd.GeoDataFrame(df_agg)
gdf_agg.set_crs(epsg=31983, inplace=True)
# gdf_agg.drop(columns=['coords']).to_file(f'results/san_remo-multipoint.gpkg', driver='GPKG')
gdf_agg.drop(columns=['coords'])
# Atualiza toda geometria com MultiPoint
# gdf_agg['geometry'] = gdf_agg['coords'].apply(MultiPoint)
# Substitui com polígonos apenas para clusters grandes
mask = gdf_agg['z_count'] >= 16

In [17]:
df_agg

,coords,z_count,z_median,z_std,z_max,geometry
ClusterID,,,,,,
1,"[[335182.75, 7396753.25], [335183.25, 7396753....",3336,9.227093,0.460550,10.333432,"MULTIPOINT ((335182.75 7396753.25), (335183.25..."
2,"[[335185.75, 7396753.25], [335186.25, 7396753....",1535,12.406150,0.669694,13.726165,"MULTIPOINT ((335185.75 7396753.25), (335186.25..."
3,"[[335193.75, 7396753.25], [335194.25, 7396753....",252,20.799744,0.201013,21.241028,"MULTIPOINT ((335193.75 7396753.25), (335194.25..."
4,"[[335203.25, 7396753.25], [335203.75, 7396753....",111,24.681206,0.133715,25.016811,"MULTIPOINT ((335203.25 7396753.25), (335203.75..."
5,"[[335209.75, 7396753.25], [335210.25, 7396753....",1121,19.095282,0.267946,20.301847,"MULTIPOINT ((335209.75 7396753.25), (335210.25..."
...,...,...,...,...,...,...
282745,"[[333863.25, 7393259.25], [333863.75, 7393259....",64,2.494175,0.302380,3.122104,"MULTIPOINT ((333863.25 7393259.25), (333863.75..."
282746,"[[333966.25, 7393258.75], [333966.75, 7393258....",32,7.285889,0.211199,7.690427,"MULTIPOINT ((333966.25 7393258.75), (333966.75..."
282750,"[[334098.75, 7393259.25], [334098.25, 7393258....",22,18.335293,0.445398,18.757166,"MULTIPOINT ((334098.75 7393259.25), (334098.25..."


In [18]:
mask = gdf_agg['z_count'] >= 16

In [19]:
gdf_agg.loc[mask, 'geometry'] = gdf_agg.loc[mask, 'geometry'].apply(lambda x: shapely.concave_hull(x, ratio=0.1, allow_holes=True))

In [20]:
# gdf_agg.loc[mask, 'geometry'] = gdf_agg.loc[mask, 'coords'].apply(lambda x: shapely.concave_hull(MultiPoint(x), ratio=0.1, allow_holes=True))

In [21]:
gdf_agg

,coords,z_count,z_median,z_std,z_max,geometry
ClusterID,,,,,,
1,"[[335182.75, 7396753.25], [335183.25, 7396753....",3336,9.227093,0.460550,10.333432,"POLYGON ((335180.25 7396750.75, 335180.75 7396..."
2,"[[335185.75, 7396753.25], [335186.25, 7396753....",1535,12.406150,0.669694,13.726165,"POLYGON ((335185.75 7396753.25, 335186.25 7396..."
3,"[[335193.75, 7396753.25], [335194.25, 7396753....",252,20.799744,0.201013,21.241028,"POLYGON ((335193.25 7396752.75, 335193.75 7396..."
4,"[[335203.25, 7396753.25], [335203.75, 7396753....",111,24.681206,0.133715,25.016811,"POLYGON ((335202.75 7396752.25, 335202.75 7396..."
5,"[[335209.75, 7396753.25], [335210.25, 7396753....",1121,19.095282,0.267946,20.301847,"POLYGON ((335209.25 7396752.75, 335209.75 7396..."
...,...,...,...,...,...,...
282745,"[[333863.25, 7393259.25], [333863.75, 7393259....",64,2.494175,0.302380,3.122104,"POLYGON ((333859.75 7393256.75, 333859.75 7393..."
282746,"[[333966.25, 7393258.75], [333966.75, 7393258....",32,7.285889,0.211199,7.690427,"POLYGON ((333965.25 7393257.75, 333965.75 7393..."
282750,"[[334098.75, 7393259.25], [334098.25, 7393258....",22,18.335293,0.445398,18.757166,"POLYGON ((334098.75 7393258.75, 334098.75 7393..."


In [22]:
gdf_agg["geometry"] = gdf_agg["geometry"].buffer(0)

In [23]:
# Suprimir warnings de buffer instável
warnings.filterwarnings("ignore", message="divide by zero encountered in buffer")

In [24]:
gdf_regularizado = regularize_geodataframe(gdf_agg[gdf_agg.area > 0].reset_index())

In [25]:
gdf_regularizado

,ClusterID,coords,z_count,z_median,z_std,z_max,geometry
0,1,"[[335182.75, 7396753.25], [335183.25, 7396753....",3336,9.227093,0.460550,10.333432,"POLYGON ((335164.879 7396732.712, 335181.83 73..."
1,2,"[[335185.75, 7396753.25], [335186.25, 7396753....",1535,12.406150,0.669694,13.726165,"POLYGON ((335185.269 7396753.113, 335191.699 7..."
2,3,"[[335193.75, 7396753.25], [335194.25, 7396753....",252,20.799744,0.201013,21.241028,"POLYGON ((335193.174 7396752.713, 335201.623 7..."
3,4,"[[335203.25, 7396753.25], [335203.75, 7396753....",111,24.681206,0.133715,25.016811,"POLYGON ((335202.415 7396750.029, 335202.701 7..."
4,5,"[[335209.75, 7396753.25], [335210.25, 7396753....",1121,19.095282,0.267946,20.301847,"POLYGON ((335209.664 7396752.759, 335218.812 7..."
...,...,...,...,...,...,...,...
45699,282745,"[[333863.25, 7393259.25], [333863.75, 7393259....",64,2.494175,0.302380,3.122104,"POLYGON ((333860.25 7393258.5, 333863.236 7393..."
45700,282746,"[[333966.25, 7393258.75], [333966.75, 7393258....",32,7.285889,0.211199,7.690427,"POLYGON ((333966.25 7393257.75, 333966.25 7393..."
45701,282750,"[[334098.75, 7393259.25], [334098.25, 7393258....",22,18.335293,0.445398,18.757166,"POLYGON ((334100.217 7393258.558, 334099.683 7..."
45702,282760,"[[333352.25, 7393258.75], [333352.75, 7393258....",30,7.112373,0.323198,7.506892,"POLYGON ((333351.873 7393258.873, 333353.305 7..."


In [26]:
# plotar 
# overlap com a geometria do distrito
gdf_regularizado = gdf_regularizado.overlay(gdf_distritos[gdf_distritos.ds_nome == 'BRAS'])

In [28]:
gdf_regularizado.loc[:, 'area_de_projecao'] = gdf_regularizado.area
gdf_regularizado.loc[:, 'gabarito'] = gdf_regularizado.loc[:, 'z_max']
gdf_regularizado.loc[:, 'pavimentos'] = gdf_regularizado.loc[:, 'z_max'] // 3.4
gdf_regularizado.loc[gdf_regularizado.z_max < 3.4, 'pavimentos'] = 1.0
gdf_regularizado.loc[:, 'area_total_construida'] = gdf_regularizado.loc[:, 'pavimentos'] * gdf_regularizado.loc[:, 'area_de_projecao']


In [29]:
colunas_manter = ['area_de_projecao', 'gabarito', 'pavimentos', 'area_total_construida', 'geometry']

In [30]:
gdf_regularizado.loc[:, colunas_manter]

,area_de_projecao,gabarito,pavimentos,area_total_construida,geometry
0,416.636128,12.432534,3.0,1249.908384,"POLYGON ((335294.361 7396547.387, 335297.368 7..."
1,23.344752,25.330460,7.0,163.413267,"POLYGON ((335280.75 7396532.75, 335282 7396532..."
2,46.434988,28.492344,8.0,371.479907,"POLYGON ((335286.042 7396532.208, 335289.875 7..."
3,52.267502,28.719328,8.0,418.140013,"POLYGON ((335311.708 7396527.993, 335317.284 7..."
4,85.540019,12.159986,3.0,256.620057,"POLYGON ((335276.013 7396527.263, 335276.446 7..."
...,...,...,...,...,...
12959,325.513082,7.041272,2.0,651.026164,"POLYGON ((335594.695 7394203.15, 335595.999 73..."
12960,2.343516,9.796299,2.0,4.687031,"POLYGON ((335516.25 7394199.25, 335515 7394200..."
12961,0.496899,10.382454,3.0,1.490696,"MULTIPOLYGON (((335203.606 7394200.574, 335203..."
12962,14.441418,5.715120,1.0,14.441418,"POLYGON ((335596.371 7394200.907, 335597.491 7..."


In [31]:
# gdf_agg = gdf_agg[gdf_agg.area > 0]

In [32]:
gdf_regularizado.loc[:, colunas_manter].reset_index().to_file(f'results/bras-multipoligono.gpkg', driver='GPKG')

In [33]:
os.remove('temp/BHM-bras.vrt')
os.remove('temp/BHM-bras.tiff')